In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor

In [2]:
class MachineLearningModel:
    def __init__(self, x_train, y_train, x_test, y_test):
        self.x_train = x_train
        self.y_train = y_train
        self.x_test = x_test
        self.y_test = y_test
        self.model = GradientBoostingRegressor(
            random_state=42,
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.9
        )
        
        self.model.fit(self.x_train, self.y_train)
        
        # --- VALUTAZIONE ---
        self.y_pred = self.model.predict(self.x_test)
        
        print("MAE test:", mean_absolute_error(self.y_test, self.y_pred))
        print("RMSE test:", np.sqrt(mean_squared_error(self.y_test, self.y_pred)))
        print("R2 test:", r2_score(self.y_test, self.y_pred))
        # --- FEATURE IMPORTANCE ---
        self.importances = pd.DataFrame({
            "feature": self.x_train.columns,
            "importance": self.model.feature_importances_
        }).sort_values(by="importance", ascending=False)
        
        print(self.importances)

    def predict(self, df_future):
        self.df_future = df_future
        # Allineamento colonne
        self.missing_cols = set(self.x_train.columns) - set(self.df_future.columns)
        for col in self.missing_cols:
            self.df_future[col] = 0
        
        self.df_future = self.df_future[self.x_train.columns]
        
        # Predizione
        self.df_future["PREDICTED_REVENUE"] = self.model.predict(self.df_future)

        return self.df_future

In [3]:
def prepare_features(df):
    df = df.copy()

    # 0. RIMOZIONE COLONNE STRETTAMENTE CORRELATE
    df = df.drop(columns=["ID_ORDER_NUM", "VAL_COST"], errors="ignore")

    # 1. Rimozione colonne descrittive
    desc_cols = [c for c in df.columns if c.startswith("DESC_")]
    df = df.drop(columns=desc_cols, errors="ignore")

    # 2. Encoding categorico compatto
    categorical_cols = [
        "ID_COMPANY", "IDS_CUSTOMER", "IDS_ITEM",
        "ID_BUSINESS_LINE", "ID_COUNTRY", "ID_AREA_MANAGER"
    ]

    enc = LabelEncoder()
    for col in categorical_cols:
        df[col] = enc.fit_transform(df[col])

    # 3. Feature temporali
    df["ID_ORDER_DATE"] = pd.to_datetime(df["ID_ORDER_DATE"])
    df["ANNO"] = df["ID_ORDER_DATE"].dt.year
    df["MESE"] = df["ID_ORDER_DATE"].dt.month
    df["GIORNO"] = df["ID_ORDER_DATE"].dt.day
    df["GIORNO_SETTIMANA"] = df["ID_ORDER_DATE"].dt.weekday
    df["WEEKEND"] = df["GIORNO_SETTIMANA"].isin([5,6]).astype(int)

    df["GIORNO_SIN"] = np.sin(2 * np.pi * df["ID_ORDER_DATE"].dt.dayofyear / 365)
    df["GIORNO_COS"] = np.cos(2 * np.pi * df["ID_ORDER_DATE"].dt.dayofyear / 365)

    # 4. ordinamento
    df = df.sort_values("ID_ORDER_DATE")

    
    return df

In [4]:
def add_lag_features(df):
    df = df.copy()

    # Ordinamento corretto
    df = df.sort_values(["IDS_CUSTOMER", "IDS_ITEM", "ID_ORDER_DATE"])
    
    group_cols = ["IDS_CUSTOMER", "IDS_ITEM"]

    # Lag sugli ordini precedenti
    df["LAG_1"] = df.groupby(group_cols)["VAL_REVENUES"].shift(1)
    df["LAG_2"] = df.groupby(group_cols)["VAL_REVENUES"].shift(2)
    df["LAG_3"] = df.groupby(group_cols)["VAL_REVENUES"].shift(3)

    # Rolling mean sugli ultimi 3 ordini
    df["ROLLING_MEAN_3"] = (
        df.groupby(group_cols)["VAL_REVENUES"]
          .transform(lambda x: x.rolling(3).mean())
    )

    # Rimuovo solo le righe senza lag
    df = df.dropna()
    df = df.drop(columns=["ID_ORDER_DATE", "ID_INVOICE_DATE"], errors="ignore")
    return df

In [5]:
def train_test_split(df):
    split_idx = int(len(df) * 0.8)
    
    train = df.iloc[:split_idx]
    test = df.iloc[split_idx:]
    
    x_train = train.drop(columns=["VAL_REVENUES"])
    y_train = train["VAL_REVENUES"]
    
    x_test = test.drop(columns=["VAL_REVENUES"])
    y_test = test["VAL_REVENUES"]

    return x_train, y_train, x_test, y_test

In [6]:
def main():
    df = pd.read_csv("SALES_OLAP.csv")
    df_ml = prepare_features(df)
    df_lag = add_lag_features(df_ml)
    x_train, y_train, x_test, y_test = train_test_split(df_lag)
    ml = MachineLearningModel(x_train, y_train, x_test, y_test)

In [7]:
if __name__ == "__main__":
    app = main()

MAE test: 1514.0531901697568
RMSE test: 1848.2030901964004
R2 test: 0.9154838383630534
             feature  importance
16    ROLLING_MEAN_3    0.343638
13             LAG_1    0.207883
14             LAG_2    0.202934
15             LAG_3    0.174531
1       IDS_CUSTOMER    0.049292
11        GIORNO_SIN    0.008268
9   GIORNO_SETTIMANA    0.006440
8             GIORNO    0.003360
12        GIORNO_COS    0.001502
7               MESE    0.001010
5    ID_AREA_MANAGER    0.000546
2           IDS_ITEM    0.000462
6               ANNO    0.000114
4         ID_COUNTRY    0.000012
0         ID_COMPANY    0.000008
3   ID_BUSINESS_LINE    0.000000
10           WEEKEND    0.000000
